# Part 1: AMR analysis, Germany vs 7 EU countries
3rd-generation cephalosporin-resistant *E. coli*, 2015-2024 (ECDC).

Run from the repository root so the `data/` folder is found. In Colab, upload the four input CSVs from `data/` first.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

## 1. Load the four input files

In [ ]:
resisantance_pct_df = pd.read_csv("data/resistance_pct_2015-2024.csv")
resistance_count_df = pd.read_csv("data/resistant_counts_2015-2024.csv")
population_df = pd.read_csv("data/population_2015-2024.csv")
incedence_df = pd.read_csv("data/ECDC_official_incidence_targets.csv")

print(resisantance_pct_df.shape, resistance_count_df.shape)

## 2. Merge percentage and counts

In [ ]:
resisantance_pct_df = resisantance_pct_df.rename(columns={
    "NumValue": "resistance_pct",
    "Indicator": "Indicator_pct",
    "Unit": "Unit_pct",
    "TxtValue": "TxtValue_pct"
})

resistance_count_df = resistance_count_df.rename(columns={
    "NumValue": "resistant_isolates",
    "Indicator": "Indicator_count",
    "Unit": "Unit_count",
    "TxtValue": "TxtValue_count"
})

AMR_df = resisantance_pct_df.merge(
    resistance_count_df,
    on=["HealthTopic", "Population", "Time", "RegionCode", "RegionName"],
    how="left"
)
print(AMR_df.shape)
print(AMR_df.columns.tolist())

## 3. Add population, calculated incidence and the official ECDC targets
Calculated incidence = resistant isolates / population x 100,000. It is used for direction and trend only. The target comparison uses the official ECDC incidence.

In [ ]:
population_df = population_df.rename(columns={
    "country": "RegionName",
    "year": "Time",
    "population": "PopulationCount"
})

AMR_df = AMR_df.merge(population_df, on=["RegionName", "Time"], how="left")

AMR_df["incidence_per_100k"] = (
    AMR_df["resistant_isolates"] / AMR_df["PopulationCount"] * 100000
).round(3)

incedence_df = incedence_df.rename(columns={"country": "RegionName"})
AMR_df = AMR_df.merge(incedence_df, on="RegionName", how="left")

print(AMR_df.shape)
print(AMR_df.isna().sum())

In [ ]:
# drop the empty text columns
AMR_df = AMR_df.drop(columns=["TxtValue_pct", "TxtValue_count"], errors="ignore")
AMR_df["year_index"] = AMR_df["Time"] - 2015

print(AMR_df.shape)
print(AMR_df.columns.tolist())
AMR_df.to_csv("AMR_final_dataset.csv", index=False)

## 4. Linear regression per country (trend direction)

In [ ]:
results = []
for country, g in AMR_df.groupby("RegionName"):
    g = g.sort_values("Time")
    X = g[["year_index"]].values
    y_pct = g["resistance_pct"].values
    y_inc = g["incidence_per_100k"].values

    m_pct = LinearRegression().fit(X, y_pct)
    m_inc = LinearRegression().fit(X, y_inc)

    results.append([
        country,
        round(m_pct.coef_[0], 3), round(r2_score(y_pct, m_pct.predict(X)), 3),
        round(m_inc.coef_[0], 3), round(r2_score(y_inc, m_inc.predict(X)), 3)
    ])

results_df = pd.DataFrame(results, columns=[
    "country", "pct_slope", "pct_r2", "incidence_slope", "incidence_r2"
])
print(results_df.sort_values("incidence_slope", ascending=False))

## 5. Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))

for country, g in AMR_df.groupby("RegionName"):
    g = g.sort_values("Time")
    axes[0].plot(g["Time"], g["resistance_pct"], marker="o", label=country)
    axes[1].plot(g["Time"], g["incidence_per_100k"], marker="o", label=country)

axes[0].set_title("% Resistant Isolates")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Resistance (%)")

axes[1].set_title("Incidence (per 100,000 population)")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Resistant isolates / 100k")
axes[1].legend(loc="upper left", bbox_to_anchor=(1,1))

fig.suptitle("% Resistant vs. Incidence: All 8 Countries (2015-2024)")
plt.tight_layout()
plt.savefig("trend_all_countries_comparison.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
for country, g in AMR_df.groupby("RegionName"):
    g = g.sort_values("Time")
    ax.plot(g["Time"], g["resistance_pct"], marker="o", label=country)

ax.set_title("3rd-Gen Cephalosporin-Resistant E. coli: % Resistant Isolates (2015-2024)")
ax.set_xlabel("Year")
ax.set_ylabel("Resistance (%)")
ax.legend(loc="upper left", bbox_to_anchor=(1,1))
plt.tight_layout()
plt.savefig("trend_all_countries_pct.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

for country in ["Germany", "Italy"]:
    g = AMR_df[AMR_df.RegionName == country].sort_values("Time")
    axes[0].plot(g["Time"], g["resistance_pct"], marker="o", label=country)
    axes[1].plot(g["Time"], g["incidence_per_100k"], marker="o", label=country)

axes[0].set_title("% Resistant Isolates")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Resistance (%)")
axes[0].legend()

axes[1].set_title("Incidence (per 100,000 population)")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Resistant isolates / 100k")
axes[1].legend()

fig.suptitle("Germany vs. Italy: % Resistant vs. Incidence Diverge (2015-2024)")
plt.tight_layout()
plt.savefig("trend_germany_italy_divergence.png", dpi=150)
plt.show()

## 6. Official ECDC targets: who is on track?

In [ ]:
target_summary = AMR_df.drop_duplicates(subset="RegionName")[[
    "RegionName", "baseline_2019", "latest_year", "latest_value",
    "change_pct", "target_2030", "target_reduction_pct", "target_type"
]].copy()

# on_track only where a 2030 target exists; Austria and Denmark have no confirmed target
target_summary["on_track"] = (target_summary["latest_value"] <= target_summary["target_2030"]).astype(object)
target_summary.loc[target_summary["target_2030"].isna(), "on_track"] = "no confirmed target"
target_summary = target_summary.sort_values("change_pct")
print(target_summary.to_string(index=False))

In [ ]:
germany = AMR_df[AMR_df.RegionName == "Germany"].sort_values("Time")

official_baseline_2019 = germany["baseline_2019"].iloc[0]
official_target_2030 = germany["target_2030"].iloc[0]
official_latest_value = germany["latest_value"].iloc[0]
official_latest_year = germany["latest_year"].iloc[0]

print(f"Germany's official 2019 incidence baseline: {official_baseline_2019}")
print(f"Germany's official 2030 target: {official_target_2030}")
print(f"Germany's most recent reported incidence ({official_latest_year}): {official_latest_value}")

on_track = official_latest_value <= official_target_2030
print(f"Already at or below target: {on_track}")

## 7. Export for the dashboard

In [ ]:
AMR_df.to_excel("AMR_dashboard_data.xlsx", index=False)
try:
    from google.colab import files
    files.download("AMR_dashboard_data.xlsx")
except ImportError:
    pass  # not running in Colab